# Independent DAS development checkpoint

This notebook reads **compact tracked products only**. It does not open raw DAS HDF5, network miniSEED, the full DAS score cache, or any held-out interval.

**Recorded outcome:** the two known local Parkfield earthquakes are the two strongest of 65 independently generated DAS triggers and have broad spatial support (8/10 and 10/10 blocks). The known regional Carpinteria network arrival is not recovered. This is promising development evidence, not a catalog-extension or family-assignment result.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
candidates = [ROOT, ROOT.parent, ROOT.parent.parent]
PROJECT = next(
    path for path in candidates
    if (path / 'outputs' / 'development_das').is_dir()
)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
DAS = PROJECT / 'outputs' / 'development_das'
NETWORK = PROJECT / 'outputs' / 'development_network'
CONFIG = PROJECT / 'config'
print('Project:', PROJECT)
print('Compact inputs only; held-out access remains sealed.')

## 1. Provenance and access gate

The raw DAS table was materialized and checksummed before any network-candidate or catalog-event time row was opened. The comparison rule was then committed separately. No raw candidate was deleted or relabeled as an earthquake.

In [ ]:
with (DAS / 'status_raw.json').open(encoding='utf-8') as handle:
    raw_status = json.load(handle)
with (DAS / 'comparison_status.json').open(encoding='utf-8') as handle:
    comparison_status = json.load(handle)
with (CONFIG / 'das_network_comparison.json').open(encoding='utf-8') as handle:
    comparison_config = json.load(handle)

raw = pd.read_csv(DAS / 'candidate_detections_raw.csv')
time_only = pd.read_csv(DAS / 'network_comparison_time_only.csv')
adjudicated = pd.read_csv(
    DAS / 'network_comparison_adjudicated.csv',
    dtype={
        'DAS_candidate_id': str,
        'network_union_candidate_id': str,
        'network_known_event_id': str,
        'DAS_catalog_event_id': str,
    },
)
network_union = pd.read_csv(
    NETWORK / 'network_candidate_union_time_only.csv'
)
network_adjudicated = pd.read_csv(
    NETWORK / 'network_candidate_union_adjudicated.csv',
    dtype={'broader_catalog_event_id': str},
)
score_preview = pd.read_csv(DAS / 'score_preview.csv')

raw_sha = hashlib.sha256(
    (DAS / 'candidate_detections_raw.csv').read_bytes()
).hexdigest()
assert raw_sha == raw_status['candidate_table_sha256']
assert raw_status['network_candidate_tables_opened'] == 0
assert raw_status['catalog_event_time_tables_opened'] == 0
assert raw_status['heldout_interval_tables_opened'] == 0
assert comparison_status['heldout_access_gate'] == 'STOP_NOT_YET_AUTHORIZED'
assert comparison_status['eligible_catalog_extension_candidate_count'] == 0

gate = pd.Series({
    'raw DAS candidates': comparison_status['raw_DAS_candidate_count'],
    'DAS + frozen-network matches': comparison_status['DAS_network_matched_event_count'],
    'known local network recovery': comparison_status['known_target_network_event_recovery'],
    'known regional network recovery': comparison_status['known_regional_network_arrival_recovery'],
    'catalog-unassociated raw DAS triggers': comparison_status['unassociated_raw_DAS_trigger_count'],
    'extension-eligible events': comparison_status['eligible_catalog_extension_candidate_count'],
    'held-out gate': comparison_status['heldout_access_gate'],
}, name='frozen result')
display(gate.to_frame())

## 2. The useful signal in version 1

The null-calibrated first-stage detector is too permissive for extension claims, but its ranking is striking. Both known local events sit far above every other trigger in score and in coherent spatial support. The plots below use the frozen rows; they do not impose a new detector.

In [ ]:
das_adjudicated = adjudicated.loc[
    adjudicated['DAS_candidate_id'].notna()
].copy()
view = raw.merge(
    das_adjudicated[[
        'DAS_candidate_id',
        'comparison_class',
        'network_known_event_id',
    ]],
    left_on='candidate_id',
    right_on='DAS_candidate_id',
    how='left',
)
view['trigger_datetime'] = pd.to_datetime(view['trigger_time'], utc=True)
known_local = view['comparison_class'].eq(
    'matched_frozen_network_known_target_event'
)
preview_time = pd.to_datetime(score_preview['trigger_time'], utc=True)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), constrained_layout=True)
axes[0].plot(
    preview_time,
    score_preview['coincidence_score'],
    color='0.65',
    linewidth=0.7,
    label='0.5 s score preview',
)
axes[0].axhline(
    raw['threshold'].iloc[0],
    color='tab:orange',
    linestyle='--',
    label='frozen v1 null threshold',
)
axes[0].scatter(
    view.loc[~known_local, 'trigger_datetime'],
    view.loc[~known_local, 'coincidence_score'],
    s=18,
    color='0.25',
    alpha=0.65,
    label='other raw triggers',
)
axes[0].scatter(
    view.loc[known_local, 'trigger_datetime'],
    view.loc[known_local, 'coincidence_score'],
    s=90,
    marker='*',
    color='crimson',
    label='known local DAS + network',
    zorder=5,
)
axes[0].set_ylabel('fourth-highest block score')
axes[0].legend(loc='upper right')
axes[0].set_title('Independent DAS score through the 50-minute development interval')

axes[1].scatter(
    view.loc[~known_local, 'block_support_count_at_declared_ratio'],
    view.loc[~known_local, 'coincidence_score'],
    color='0.35',
    alpha=0.65,
    label='other raw triggers',
)
axes[1].scatter(
    view.loc[known_local, 'block_support_count_at_declared_ratio'],
    view.loc[known_local, 'coincidence_score'],
    s=110,
    marker='*',
    color='crimson',
    label='known local DAS + network',
)
axes[1].set_xlabel('blocks at declared characteristic ratio >= 2')
axes[1].set_ylabel('fourth-highest block score')
axes[1].set_xticks(range(11))
axes[1].legend(loc='upper left')
axes[1].set_title('Spatial-support separation')
plt.show()

display(view.sort_values('coincidence_score', ascending=False)[[
    'candidate_id',
    'trigger_time',
    'coincidence_score',
    'block_support_count_at_declared_ratio',
    'network_known_event_id',
    'comparison_class',
]].head(10))

## 3. Advisor sandbox (in memory only)

Change the three values below and rerun the cell. The matching window and display filters are exploratory: they never overwrite the frozen 8-second comparison or the 65-candidate raw table. The default support filter reflects the already registered ratio-of-2 and four-block concepts, but applying it as a hard gate would be a **development-tuned version 2**, not an independent validation result.

In [ ]:
from src.das_network_comparison import build_time_only_comparison

EXPLORATORY_MATCH_WINDOW_S = 8.0
EXPLORATORY_MIN_SCORE = 2.0
EXPLORATORY_MIN_STRONG_BLOCKS = 4

sandbox_union = pd.DataFrame(build_time_only_comparison(
    raw.to_dict('records'),
    network_union.to_dict('records'),
    maximum_difference_s=EXPLORATORY_MATCH_WINDOW_S,
))
sandbox_selected = view.loc[
    (view['coincidence_score'] >= EXPLORATORY_MIN_SCORE)
    & (view['block_support_count_at_declared_ratio'] >= EXPLORATORY_MIN_STRONG_BLOCKS)
].copy()

display(pd.Series({
    'time-only DAS + network matches': int((sandbox_union['comparison_membership'] == 'DAS+network').sum()),
    'time-only DAS-only rows': int((sandbox_union['comparison_membership'] == 'DAS_only').sum()),
    'time-only network-only rows': int((sandbox_union['comparison_membership'] == 'network_only').sum()),
    'raw DAS rows passing exploratory display filters': len(sandbox_selected),
}, name='exploratory only').to_frame())
display(sandbox_selected[[
    'candidate_id',
    'trigger_time',
    'coincidence_score',
    'block_support_count_at_declared_ratio',
    'network_known_event_id',
    'comparison_class',
]].sort_values('coincidence_score', ascending=False))

## 4. Catalog audit without overclaiming

A broad physical travel-time window makes 11 raw DAS triggers compatible with five catalog events. Six are secondary triggers for an already represented event. Three representative unmatched triggers are compatible with distant regional events, but each has zero strong-block support; compatibility alone is not proof that DAS detected those events. Fifty-four triggers have no catalog association and require waveform/spatial morphology review.

In [ ]:
display(
    adjudicated['comparison_class']
    .value_counts()
    .rename_axis('comparison class')
    .to_frame('row count')
)
representative = adjudicated.loc[
    adjudicated['catalog_event_representative']
    .astype(str).str.lower().eq('true')
].copy()
display(representative[[
    'DAS_candidate_id',
    'DAS_trigger_time',
    'DAS_coincidence_score',
    'DAS_block_support_count_at_declared_ratio',
    'DAS_catalog_event_id',
    'DAS_catalog_location_name',
    'DAS_catalog_horizontal_distance_km',
    'comparison_class',
]].sort_values('DAS_trigger_time'))

## Checkpoint decision

- **Worth pursuing:** an independent DAS-only scan recovered both known local events, ranked them first and second, and did not recover the network's distant regional arrival. That is an initial demonstration of near-fiber selectivity.
- **Not yet an extension:** the interval was deliberately nonblind and contains only two known local positives; version 1 also emitted 63 unmatched raw triggers. There are zero validated DAS-only catalog additions and zero family assignments.
- **Concrete next design:** freeze a version-2 spatial-coherence gate before held-out access, motivated transparently by this development result. A natural hypothesis is to require at least four blocks at the already declared characteristic ratio of 2, while retaining the v1 table unchanged.
- **Validation question:** on sealed intervals, compare network-only and DAS-only event recovery and false discoveries at the event level. Only DAS detections that survive blinded morphology review and are absent from the full network union can establish catalog extension.
- **Repeater question after detection:** use independently known family populations to test whether DAS improves family partitioning. Creep rate and stress drop remain downstream until detection and geometry are validated.